# ROGII Colab setup

This notebook mounts Drive, checks out the repository, installs the locked environment, copies the input CSVs to `/content`, and validates the competition data. Use a Python 3.12 Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set this to your Git repository. For a private repository, use Colab Secrets or SSH; never paste a token into the notebook.
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
REVISION = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/kaggle/rogii'
LOCAL_DATA_DIR = '/content/rogii-data'

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/ROGII')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--all', '--prune'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', REVISION], check=True)
print(subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
import os
import sys

assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version}'
subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
os.chdir(repo_dir)
subprocess.run(['uv', 'venv', '--python', sys.executable, '--system-site-packages'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], check=True)
subprocess.run(['uv', 'run', 'python', '--version'], check=True)

In [ ]:
import shutil

drive_data = Path(DRIVE_ROOT) / 'data'
local_data = Path(LOCAL_DATA_DIR)
assert (drive_data / 'train').is_dir(), 'Missing train directory in Google Drive'
assert (drive_data / 'test').is_dir(), 'Missing test directory in Google Drive'
shutil.copytree(drive_data, local_data, dirs_exist_ok=True)
os.environ['ROGII_DATA_DIR'] = str(local_data)
os.environ['ROGII_ARTIFACT_DIR'] = str(Path(DRIVE_ROOT) / 'artifacts')
print('data:', os.environ['ROGII_DATA_DIR'])
print('artifacts:', os.environ['ROGII_ARTIFACT_DIR'])

In [ ]:
# First validate a small subset. Remove the two smoke-test flags for the complete 773-well preparation.
subprocess.run([
    'uv', 'run', 'rogii-prepare',
    '--data-dir', os.environ['ROGII_DATA_DIR'],
    '--artifact-dir', os.environ['ROGII_ARTIFACT_DIR'],
    '--limit-wells', '8',
    '--skip-fingerprint',
], check=True)

## Run the baseline smoke test

This uses eight wells and writes the result to Drive. After it succeeds, choose a new run ID and remove `--limit-wells 8` for the full 773-well run.

In [ ]:
import json

RUN_ID = 'baseline_anchor_smoke_v001'
subprocess.run([
    'uv', 'run', 'rogii-train',
    '--config', 'configs/experiment/baseline_anchor.yaml',
    '--run-id', RUN_ID,
    '--data-dir', os.environ['ROGII_DATA_DIR'],
    '--artifact-dir', os.environ['ROGII_ARTIFACT_DIR'],
    '--limit-wells', '8',
], check=True)
run_dir = Path(os.environ['ROGII_ARTIFACT_DIR']) / RUN_ID
json.loads((run_dir / 'metrics.json').read_text())